Hyperparams selection

# Libraries

In [1]:
import openpyxl
import pandas as pd
from pathlib import Path
import numpy as np
import sys
from tqdm import tqdm
import optuna
import joblib
import json
from pathlib import Path
import os
from sklearn.model_selection import cross_val_score
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [3]:
# hyperparams
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import *

In [4]:
# sklearn models pipeline
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_utils import MLPipeline

In [5]:
RANDOM_STATE = 42

# Datatasets selection

In [6]:
path_data = Path('../data')
path_results = Path('../results/')
path_highlighted = path_results / 'metrics_modelling2_highlight.xlsx'

In [7]:
df_filtered = pd.DataFrame()
for i, page in enumerate(['Net Impact', 'Splashing']):
    df_highlighted = pd.read_excel(path_highlighted, sheet_name=page)
    wb = openpyxl.load_workbook(path_highlighted, data_only=True)
    ws = wb.worksheets[i]

    highlighted_rows = []
    for row in range(1, ws.max_row):
        cell = ws.cell(column=1, row=row)
        bgColor = cell.fill.bgColor.index
        fgColor = cell.fill.fgColor.index
        if bgColor != '00000000':
            highlighted_rows.append(row)
    highlighted_rows = np.array(highlighted_rows) - 2
    df_temp = df_highlighted.iloc[highlighted_rows]
    df_filtered = pd.concat((df_filtered, df_temp))
df_filtered.reset_index(drop=True, inplace=True)

In [8]:
# svc kernels addition
df_svc = df_filtered.loc[
    df_filtered['model'].apply(lambda x: 'svc' in x)].copy()
df_svc_linear, df_svc_poly = df_svc.copy(), df_svc.copy()
df_svc_linear['kernel'] = 'linear'
df_svc_poly['kernel'] = 'poly'
df_filtered_svc = pd.concat(
    (df_filtered.loc[df_filtered['model'].apply(lambda x: 'svc' not in x)],
     df_svc_linear, df_svc_poly
     ))
df_filtered_svc.reset_index(drop=True, inplace=True)

Datasets for parameters selection.

In [9]:
df_filtered_svc

,dataset,target,model,accuracy,f1,precision,recall,roc_auc,kernel
0,df_modelling_dimensionless,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.900000,0.931818,NaN
1,df_modelling_dimensionless_pf,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.900000,0.931818,NaN
2,df_modelling_dimensionless,net_impact,catboostclassifier_smote_net_impact,0.933333,0.878049,0.857143,0.900000,0.922727,NaN
3,df_modelling_no_multicollinearity,net_impact,kneighborsclassifier_net_impact_ordenc,0.933333,0.878049,0.857143,0.900000,0.922727,NaN
4,df_modelling_no_multicollinearity,net_impact,xgbclassifier_smote_net_impact,0.933333,0.871795,0.894737,0.850000,0.906818,NaN
5,df_modelling_no_multicollinearity_pf,net_impact,xgbclassifier_ohe_smote_net_impact,0.933333,0.871795,0.894737,0.850000,0.906818,NaN
6,df_modelling_no_multicollinearity_pf,net_impact,kneighborsclassifier_net_impact_onehot,0.920000,0.857143,0.818182,0.900000,0.913636,NaN
7,df_modelling_no_multicollinearity,net_impact,xgbclassifier_ohe_smote_net_impact,0.920000,0.850000,0.850000,0.850000,0.897727,NaN
8,df_modelling_no_multicollinearity_pf,net_impact,logisticregression_net_impact_ordenc,0.906667,0.820513,0.842105,0.800000,0.872727,NaN
9,df_modelling_dimensionless,net_impact,logisticregression_net_impact_ordenc,0.893333,0.800000,0.800000,0.800000,0.863636,NaN


In [10]:
sorted(df_filtered_svc['model'].unique())

['catboostclassifier_net_impact',
 'catboostclassifier_smote_net_impact',
 'catboostclassifier_smote_splashing',
 'catboostclassifier_splashing',
 'kneighborsclassifier_net_impact_onehot',
 'kneighborsclassifier_net_impact_ordenc',
 'kneighborsclassifier_smote_net_impact_ordenc',
 'kneighborsclassifier_smote_splashing_ordenc',
 'kneighborsclassifier_splashing_ordenc',
 'logisticregression_net_impact_ordenc',
 'logisticregression_splashing_onehot',
 'svc_smote_net_impact_onehot',
 'svc_smote_net_impact_ordenc',
 'svc_smote_splashing_onehot',
 'svc_smote_splashing_ordenc',
 'xgbclassifier_ohe_smote_net_impact',
 'xgbclassifier_ohe_smote_splashing',
 'xgbclassifier_smote_net_impact',
 'xgbclassifier_smote_splashing']

# Parameters selection

In [11]:
path_best_model = Path('../results/best_models_modelling_2')
if not os.path.exists(path_best_model): os.makedirs(path_best_model)

In [12]:
def objective(
        trial, train, target, model_str, 
        random_state, dataset_filename, kernel, cat_features=['wettability']):
    params = get_params(trial, model_str, random_state, cat_features)
    if 'svc' in model_str: params['kernel'] = kernel
    model = get_model(model_str, params)
    
    preproc_pipeline = get_preproc_pipeline(model_str=model_str, random_state=random_state, 
                                            cat_features=cat_features, 
                                            num_features=dict_num_features[dataset_filename])
    pipeline = [('model', model)]
    if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
    pipeline = Pipeline(steps=pipeline)
    mean_f1 = cross_val_score(pipeline, X=train.drop(columns=[target]), 
                              y=train[target], cv=5, scoring='f1').mean()
    return mean_f1

In [13]:
path_df_results = '../results/metrics_modelling2.xlsx'
df_results = pd.read_excel(path_df_results)
df_results['optuna_flg'] = 0

In [14]:
df_optuna = pd.DataFrame()
dict_results = {}
for i in (pbar:=tqdm(range(df_filtered_svc.shape[0]))):
    dataset_filename = df_filtered_svc.iloc[i]['dataset']
    if 'df_full' in dataset_filename: continue
    model_str = df_filtered_svc.iloc[i]['model']
    kernel = ''
    if 'svc' in model_str: kernel = df_filtered_svc.iloc[i]['kernel']
    pbar.set_description(f"Processing\t{model_str}\t{dataset_filename}\t{kernel}")
    target = df_filtered_svc.iloc[i]['target']
    train, test = get_train_test(
        dataset_filename=dataset_filename,
        target=target)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    def obj(trial):
        return objective(
            trial, train, 
            target, model_str, dataset_filename=dataset_filename, 
            random_state=RANDOM_STATE, cat_features=['wettability'], kernel=kernel)
    study.optimize(obj, n_trials=400, show_progress_bar=True)#, n_jobs=os.cpu_count()-2)
    params = study.best_params
    if 'svc' in model_str: params['probability'] = True
    model = get_model(model_str, params)
    preproc_pipeline = get_preproc_pipeline(model_str=model_str, random_state=RANDOM_STATE, 
                                            cat_features=['wettability'], 
                                            num_features=dict_num_features[dataset_filename])
    preproc_pipeline = [x for x in preproc_pipeline if x[0]!='smt']
    pipeline = [('model', model)]
    if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
    pipeline = Pipeline(steps=pipeline)
    pipeline.fit(train.drop(columns=[target]), train[target])
    preds = pipeline.predict(test.drop(columns=[target]))
    model_fname = f'{model_str}_{dataset_filename}'
    if 'svc' in model_str:  model_fname += f'_{kernel}'
    joblib.dump(pipeline, str(path_best_model / f'{model_fname}.pkl'))
    metrics = [{
            'accuracy': accuracy_score(test[target], preds),
            'f1': f1_score(test[target], preds),
            'precision': precision_score(test[target], preds),
            'recall': recall_score(test[target], preds),
            'roc_auc': roc_auc_score(test[target], preds)}]
    dict_results[f'{model_str}_{dataset_filename}'] = {
            'dataset': dataset_filename,
            'model': model_str,
            'target': target,
            'metrics': metrics,
            'params': study.best_params}
    if 'svc' in model_str: model_str += f'_{kernel}'
    df_current = pd.DataFrame({
            'dataset': [dataset_filename],
            'target': [target],
            'model': [f'{model_str}'.replace(f'_{dataset_filename}', '')],
            'optuna_flg': [1],
            'accuracy': [metrics[0]['accuracy']],
            'f1': [metrics[0]['f1']],
            'precision': [metrics[0]['precision']],
            'recall': [metrics[0]['recall']],
            'roc_auc': [metrics[0]['roc_auc']]})
    df_optuna = pd.concat((df_optuna, df_current))
    clear_output()
    metrics = []
df_results = pd.concat((df_filtered, df_optuna))

Processing	svc_smote_splashing_ordenc	df_modelling_no_multicollinearity_pf	poly: 100%|██████████| 39/39 [8:11:13<00:00, 755.74s/it]


In [15]:
with open("../results/optuna_results.json", "w") as outfile: 
    json.dump(dict_results, outfile)
df_results.reset_index(drop=True, inplace=True)
df_results.to_excel('../results/metrics_modelling2_filtered_optuna.xlsx')